# CIFAR-10 Single Instance Overfitting Verification

This notebook verifies that the model successfully overfits a single CIFAR-10 image.

**Expected behavior:**
- Generated images should closely match the target image
- MSE between generated and target should be very low (<0.01)
- Visual comparison should show near-identical images

## Setup

### Training command:
```bash
python train_cifar10_single.py --max_steps 5000 --image_index 0
```

In [ ]:
# Navigate to PixNerd folder
import os
import sys

NOTEBOOK_DIR = os.getcwd()
print(f"Starting directory: {NOTEBOOK_DIR}")

# Navigate to PixNerd folder (where src/ lives)
PIXNERD_DIR = os.path.join(NOTEBOOK_DIR, "PixNerd")
if os.path.exists(PIXNERD_DIR):
    os.chdir(PIXNERD_DIR)
    print(f"Changed to: {os.getcwd()}")
elif os.path.basename(NOTEBOOK_DIR) == "PixNerd":
    print(f"Already in PixNerd directory: {NOTEBOOK_DIR}")
else:
    parent = os.path.dirname(NOTEBOOK_DIR)
    pixnerd_in_parent = os.path.join(parent, "PixNerd")
    if os.path.exists(pixnerd_in_parent):
        os.chdir(pixnerd_in_parent)
        print(f"Changed to: {os.getcwd()}")
    else:
        print(f"WARNING: Could not find PixNerd folder")

if os.path.exists("src"):
    print("Found src/ directory")
else:
    print("ERROR: src/ directory not found!")

In [ ]:
from pathlib import Path
import math
import numpy as np
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
from PIL import Image

# Paths
PIXNERD_ROOT = Path(os.getcwd())

# ============================================================
# CHECKPOINT PATH - UPDATE THIS TO YOUR TRAINED MODEL
# ============================================================
EXP_DIR = PIXNERD_ROOT / "workdirs" / "exp_cifar10_single_overfit"
CKPT_PATH = EXP_DIR / "checkpoints" / "last.ckpt"
TARGET_IMG_PATH = EXP_DIR / "target_image.png"
CLASS_LABEL_PATH = EXP_DIR / "training_class.txt"
# ============================================================

OUTPUT_DIR = PIXNERD_ROOT / "outputs" / "cifar10_single"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# Model config (must match train_cifar10_single.py)
NUM_CLASSES = 10
BASE_RES = 32
PATCH_SIZE = 8
HIDDEN_SIZE = 512
DECODER_HIDDEN_SIZE = 64
NUM_ENCODER_BLOCKS = 8
NUM_DECODER_BLOCKS = 2
NUM_GROUPS = 8

CIFAR10_CLASSES = [
    'airplane', 'automobile', 'bird', 'cat', 'deer',
    'dog', 'frog', 'horse', 'ship', 'truck'
]

print(f"Checkpoint path: {CKPT_PATH}")
print(f"Checkpoint exists: {CKPT_PATH.exists()}")
print(f"Target image exists: {TARGET_IMG_PATH.exists()}")
print(f"Class label file exists: {CLASS_LABEL_PATH.exists()}")
print(f"Device: {DEVICE}")

## Load Target Image

In [ ]:
# Load target image
target_pil = Image.open(TARGET_IMG_PATH)
target_np = np.array(target_pil)
target_tensor = torch.from_numpy(target_np).permute(2, 0, 1).float() / 255.0
target_tensor = (target_tensor - 0.5) / 0.5  # Normalize to [-1, 1]

print(f"Target image shape: {target_np.shape}")
print(f"Target tensor shape: {target_tensor.shape}")

plt.figure(figsize=(4, 4))
plt.imshow(target_np)
plt.title("Target Image (to be memorized)")
plt.axis('off')
plt.show()

## Build Model

In [ ]:
# Import PixNerd components
from src.models.autoencoder.pixel import PixelAE
from src.models.conditioner.class_label import LabelConditioner
from src.models.transformer.pixnerd_c2i_heavydecoder import PixNerDiT
from src.diffusion.flow_matching.scheduling import LinearScheduler
from src.diffusion.flow_matching.sampling import EulerSampler, ode_step_fn
from src.diffusion.base.guidance import simple_guidance_fn
from src.diffusion.flow_matching.training import FlowMatchingTrainer
from src.callbacks.simple_ema import SimpleEMA
from src.lightning_model import LightningModel
from src.models.autoencoder.base import fp2uint8

print("Imports successful!")

In [ ]:
print("Initializing model components...")

main_scheduler = LinearScheduler()

vae = PixelAE(scale=1.0)

conditioner = LabelConditioner(num_classes=NUM_CLASSES)

# Create denoiser directly (we'll load EMA weights into this)
denoiser = PixNerDiT(
    in_channels=3,
    patch_size=PATCH_SIZE,
    num_groups=NUM_GROUPS,
    hidden_size=HIDDEN_SIZE,
    decoder_hidden_size=DECODER_HIDDEN_SIZE,
    num_encoder_blocks=NUM_ENCODER_BLOCKS,
    num_decoder_blocks=NUM_DECODER_BLOCKS,
    num_classes=NUM_CLASSES,
)

# Sampler with low guidance for overfitting
sampler = EulerSampler(
    num_steps=50,
    guidance=1.0,  # No CFG for overfitting
    guidance_interval_min=0.0,
    guidance_interval_max=1.0,
    scheduler=main_scheduler,
    w_scheduler=LinearScheduler(),
    guidance_fn=simple_guidance_fn,
    step_fn=ode_step_fn,
)

print(f"Denoiser parameters: {sum(p.numel() for p in denoiser.parameters()):,}")
print("Model components initialized!")

## Load Checkpoint

In [ ]:
print(f"Loading checkpoint from: {CKPT_PATH}")
ckpt = torch.load(CKPT_PATH, map_location=DEVICE, weights_only=False)

# Analyze checkpoint structure
state_dict = ckpt["state_dict"]
print(f"\nCheckpoint keys: {len(state_dict)}")

# Count keys by prefix
prefixes = {}
for key in state_dict.keys():
    prefix = key.split('.')[0]
    prefixes[prefix] = prefixes.get(prefix, 0) + 1
print(f"Key prefixes: {prefixes}")

# Show sample keys for each prefix
print(f"\nSample keys by prefix:")
for prefix in sorted(prefixes.keys()):
    prefix_keys = [k for k in state_dict.keys() if k.startswith(prefix + '.')][:2]
    for key in prefix_keys:
        print(f"  {key}: shape={state_dict[key].shape}")

In [ ]:
# Load EMA denoiser weights directly into our denoiser
# The checkpoint saves with prefix "ema_denoiser." so we need to strip it

print("Loading EMA denoiser weights...")

# Extract EMA denoiser weights
ema_prefix = "ema_denoiser."
ema_weights = {}
for key, value in state_dict.items():
    if key.startswith(ema_prefix):
        new_key = key[len(ema_prefix):]
        ema_weights[new_key] = value

print(f"Found {len(ema_weights)} EMA denoiser keys")

# Check if keys match
denoiser_keys = set(denoiser.state_dict().keys())
ema_keys = set(ema_weights.keys())

missing = denoiser_keys - ema_keys
unexpected = ema_keys - denoiser_keys

print(f"Missing in checkpoint: {len(missing)}")
print(f"Unexpected in checkpoint: {len(unexpected)}")

if missing:
    print(f"  Missing keys: {list(missing)[:5]}...")
if unexpected:
    print(f"  Unexpected keys: {list(unexpected)[:5]}...")

# Load the weights
result = denoiser.load_state_dict(ema_weights, strict=False)
print(f"\nload_state_dict result:")
print(f"  Missing: {len(result.missing_keys)}")
print(f"  Unexpected: {len(result.unexpected_keys)}")

# Move to device and ensure float32 (EMA is kept in float32 during training)
denoiser = denoiser.to(DEVICE).float()  # Explicit float32
denoiser.eval()

# Verify model precision
sample_param = next(denoiser.parameters())
print(f"\nModel precision: {sample_param.dtype}")
print(f"Model device: {sample_param.device}")

# Verify weights are loaded (check a sample weight)
print("\nVerifying loaded weights:")
if hasattr(denoiser, 'blocks') and len(denoiser.blocks) > 0:
    sample_weight = denoiser.blocks[0].norm1.weight
    print(f"  blocks[0].norm1.weight: shape={sample_weight.shape}")
    print(f"    mean={sample_weight.mean().item():.6f}")
    print(f"    std={sample_weight.std().item():.6f}")
    print(f"    min={sample_weight.min().item():.6f}")
    print(f"    max={sample_weight.max().item():.6f}")
    
    # For RMSNorm, default weight is all 1s - check if we loaded different values
    if torch.allclose(sample_weight, torch.ones_like(sample_weight), atol=1e-4):
        print("    WARNING: Weight appears to be at initialization (all 1s)!")
    else:
        print("    Weights appear to be loaded correctly (not default values)")

# Also check class embedding (LabelEmbedder is named y_embedder in PixNerDiT)
if hasattr(denoiser, 'y_embedder'):
    class_weight = denoiser.y_embedder.embedding_table.weight
    print(f"\n  y_embedder.embedding_table.weight: shape={class_weight.shape}")
    print(f"    mean={class_weight.mean().item():.6f}")
    print(f"    std={class_weight.std().item():.6f}")
    print(f"    Expected shape: [{NUM_CLASSES + 1}, {HIDDEN_SIZE}] = [11, 512]")

print("\nCheckpoint loaded successfully!")

# Quick forward pass test
print("\nTesting forward pass...")
with torch.no_grad():
    test_noise = torch.randn(1, 3, 32, 32, device=DEVICE)
    test_t = torch.tensor([0.5], device=DEVICE)
    test_class = torch.tensor([0], device=DEVICE)  # Class 0
    
    try:
        test_output = denoiser(test_noise, test_t, test_class)
        print(f"  Input shape: {test_noise.shape}")
        print(f"  Output shape: {test_output.shape}")
        print(f"  Output range: [{test_output.min().item():.3f}, {test_output.max().item():.3f}]")
        print("  Forward pass successful!")
    except Exception as e:
        print(f"  Forward pass failed: {e}")

## Helper Functions

In [ ]:
@torch.no_grad()
def sample_single(
    class_label: int,
    num_samples: int = 1,
    seed: int = 42,
    num_steps: int = 50,
    guidance: float = 1.0,
    use_autocast: bool = True,
):
    """Generate samples for a specific class.
    
    Args:
        class_label: CIFAR-10 class index (0-9)
        num_samples: Number of samples to generate
        seed: Random seed for reproducibility
        num_steps: Number of sampling steps
        guidance: CFG guidance scale (1.0 = no guidance)
        use_autocast: Whether to use bfloat16 autocast (default True, set False for debugging)
    """
    torch.manual_seed(seed)
    
    # Configure sampler
    sampler.guidance = guidance
    sampler.num_steps = num_steps
    
    # Generate noise
    noise = torch.randn(num_samples, 3, 32, 32, device=DEVICE)
    
    # Get condition
    labels = [class_label] * num_samples
    condition, uncondition = conditioner(labels)
    condition = condition.to(DEVICE)
    uncondition = uncondition.to(DEVICE)
    
    # Sample using the denoiser directly
    # NOTE: The sampler has @torch.autocast("cuda", dtype=torch.bfloat16) decorator
    # which may cause precision issues. For debugging, we can bypass this.
    if use_autocast:
        samples = sampler(
            denoiser,
            noise,
            condition,
            uncondition,
        )
    else:
        # Manual sampling without autocast for debugging precision issues
        with torch.cuda.amp.autocast(enabled=False):
            samples = sampler._impl_sampling(denoiser, noise.float(), condition, uncondition)[0][-1]
    
    # Decode
    images = vae.decode(samples)
    images = torch.clamp(images, -1.0, 1.0)
    images_uint8 = fp2uint8(images)
    
    return images_uint8.cpu(), samples.cpu()


def compute_metrics(generated, target):
    """Compute MSE and PSNR between generated and target images."""
    # Convert to [0, 1] range
    if generated.max() > 1:
        generated = generated.float() / 255.0
    if target.max() > 1:
        target = target.float() / 255.0
    
    mse = F.mse_loss(generated, target).item()
    psnr = 10 * np.log10(1.0 / (mse + 1e-10))
    
    return mse, psnr


@torch.no_grad()
def compute_training_loss_at_t(target_img, class_label, t_value=0.5):
    """
    Compute what the training loss would be for the target image at timestep t.
    This helps verify the model's behavior matches training.
    """
    # Prepare inputs like training
    x = target_img.unsqueeze(0).to(DEVICE)  # [1, 3, 32, 32]
    t = torch.tensor([t_value], device=DEVICE)
    y = torch.tensor([class_label], device=DEVICE)
    
    # Create noise
    noise = torch.randn_like(x)
    
    # Compute alpha, sigma for LinearScheduler
    alpha = t.view(-1, 1, 1, 1)  # t
    sigma = (1 - t).view(-1, 1, 1, 1)  # 1-t
    
    # Create noisy sample: x_t = alpha * x + sigma * noise
    x_t = alpha * x + sigma * noise
    
    # Target velocity: v_t = dalpha * x + dsigma * noise = x - noise
    v_t = x - noise
    
    # Get model prediction
    v_pred = denoiser(x_t, t, y)
    
    # Compute loss
    loss = F.mse_loss(v_pred, v_t)
    
    return loss.item(), x_t, v_t, v_pred


print("Helper functions defined.")
print("\nAdditional diagnostic function: compute_training_loss_at_t()")
print("  - This verifies the model's forward pass matches training expectations")

## Find Correct Class Label

If the class label file doesn't exist, we'll test all 10 classes and find the one with lowest MSE.

In [ ]:
# Try to load training class from file
if CLASS_LABEL_PATH.exists():
    with open(CLASS_LABEL_PATH) as f:
        CLASS_LABEL = int(f.read().strip())
    print(f"Loaded class label from file: {CLASS_LABEL} ({CIFAR10_CLASSES[CLASS_LABEL]})")
else:
    print("Class label file not found. Testing all 10 classes to find the best match...")
    print()
    
    tgt_tensor = torch.from_numpy(target_np).permute(2, 0, 1).float() / 255.0
    
    class_mses = []
    for class_idx in range(10):
        samples_uint8, _ = sample_single(
            class_label=class_idx,
            num_samples=1,
            seed=42,
            num_steps=50,
            guidance=1.0,
        )
        gen_tensor = samples_uint8[0].float() / 255.0
        mse, _ = compute_metrics(gen_tensor, tgt_tensor)
        class_mses.append(mse)
        print(f"  Class {class_idx} ({CIFAR10_CLASSES[class_idx]:>10}): MSE = {mse:.6f}")
    
    CLASS_LABEL = np.argmin(class_mses)
    print()
    print(f"Best class: {CLASS_LABEL} ({CIFAR10_CLASSES[CLASS_LABEL]}) with MSE = {class_mses[CLASS_LABEL]:.6f}")

print(f"\nUsing class label: {CLASS_LABEL} ({CIFAR10_CLASSES[CLASS_LABEL]})")

In [ ]:
# Diagnostic: Verify training loss computation at inference time
# If the model truly memorized the image, the training loss should be very low

print("="*60)
print("DIAGNOSTIC: Verifying training loss at inference time")
print("="*60)

# Load target in the correct format for loss computation
target_for_loss = target_tensor.to(DEVICE)

# Test at multiple timesteps
timesteps_to_test = [0.1, 0.3, 0.5, 0.7, 0.9]
losses = []

print(f"\nComputing training loss for class {CLASS_LABEL} ({CIFAR10_CLASSES[CLASS_LABEL]}):")
for t_val in timesteps_to_test:
    torch.manual_seed(42)  # Fixed seed for reproducibility
    loss, x_t, v_t, v_pred = compute_training_loss_at_t(target_for_loss, CLASS_LABEL, t_val)
    losses.append(loss)
    print(f"  t={t_val:.1f}: loss={loss:.6f}")

avg_loss = np.mean(losses)
print(f"\nAverage loss across timesteps: {avg_loss:.6f}")

if avg_loss < 0.001:
    print("EXCELLENT: Model has very low training loss - weights are correctly loaded!")
elif avg_loss < 0.01:
    print("GOOD: Model has low training loss - should produce good reconstructions.")
elif avg_loss < 0.1:
    print("WARNING: Model has moderate loss - may not have fully memorized the image.")
else:
    print("ERROR: Model has high loss - weights may not be correctly loaded!")
    print("  Check that you're loading the EMA weights, not the online model weights.")

print("="*60)

## Generate and Compare

Generate samples with different seeds and compare to target.

In [ ]:
# Generate multiple samples - compare autocast vs full precision
num_samples = 5
all_samples = []
all_mses = []
all_psnrs = []
all_samples_nocast = []
all_mses_nocast = []
all_psnrs_nocast = []

tgt_tensor = torch.from_numpy(target_np).permute(2, 0, 1).float() / 255.0

print("Generating samples with autocast (bfloat16):")
for seed in range(num_samples):
    samples_uint8, samples_raw = sample_single(
        class_label=CLASS_LABEL,
        num_samples=1,
        seed=seed,
        num_steps=100,  # More steps for better quality
        guidance=1.0,
        use_autocast=True,
    )
    all_samples.append(samples_uint8[0])
    
    # Compute metrics
    gen_tensor = samples_uint8[0].float() / 255.0
    mse, psnr = compute_metrics(gen_tensor, tgt_tensor)
    all_mses.append(mse)
    all_psnrs.append(psnr)
    print(f"  Seed {seed}: MSE={mse:.6f}, PSNR={psnr:.2f} dB")

print(f"\nAutocast - Average MSE: {np.mean(all_mses):.6f}, Average PSNR: {np.mean(all_psnrs):.2f} dB")

# Also test without autocast for comparison (if on CUDA)
if DEVICE == "cuda":
    print("\nGenerating samples WITHOUT autocast (float32):")
    for seed in range(num_samples):
        samples_uint8, samples_raw = sample_single(
            class_label=CLASS_LABEL,
            num_samples=1,
            seed=seed,
            num_steps=100,
            guidance=1.0,
            use_autocast=False,
        )
        all_samples_nocast.append(samples_uint8[0])
        
        gen_tensor = samples_uint8[0].float() / 255.0
        mse, psnr = compute_metrics(gen_tensor, tgt_tensor)
        all_mses_nocast.append(mse)
        all_psnrs_nocast.append(psnr)
        print(f"  Seed {seed}: MSE={mse:.6f}, PSNR={psnr:.2f} dB")
    
    print(f"\nNo autocast - Average MSE: {np.mean(all_mses_nocast):.6f}, Average PSNR: {np.mean(all_psnrs_nocast):.2f} dB")
    
    # Compare
    print(f"\nPrecision impact: MSE diff = {np.mean(all_mses) - np.mean(all_mses_nocast):.6f}")
else:
    print("\n(Skipping no-autocast comparison - not on CUDA)")
    all_samples_nocast = all_samples
    all_mses_nocast = all_mses
    all_psnrs_nocast = all_psnrs

In [ ]:
# Visual comparison
fig, axes = plt.subplots(2, num_samples + 1, figsize=(3 * (num_samples + 1), 6))

# Top row: target and generated samples
axes[0, 0].imshow(target_np)
axes[0, 0].set_title("Target")
axes[0, 0].axis('off')

for i, sample in enumerate(all_samples):
    sample_np = sample.permute(1, 2, 0).numpy()
    axes[0, i + 1].imshow(sample_np)
    axes[0, i + 1].set_title(f"Seed {i}\nMSE={all_mses[i]:.4f}")
    axes[0, i + 1].axis('off')

# Bottom row: difference maps
axes[1, 0].axis('off')
axes[1, 0].set_title("Difference Maps")

for i, sample in enumerate(all_samples):
    sample_np = sample.permute(1, 2, 0).numpy().astype(float)
    diff = np.abs(sample_np - target_np.astype(float))
    diff_normalized = (diff / 255.0 * 5).clip(0, 1)  # Amplify for visibility
    axes[1, i + 1].imshow(diff_normalized)
    axes[1, i + 1].set_title(f"Diff (5x amp)")
    axes[1, i + 1].axis('off')

plt.suptitle(f"Single Instance Overfitting - Class: {CIFAR10_CLASSES[CLASS_LABEL]}", fontsize=14)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "overfitting_comparison.png", dpi=150)
plt.show()

## Metrics Summary

In [ ]:
print("="*60)
print("OVERFITTING VERIFICATION SUMMARY")
print("="*60)
print(f"Class: {CLASS_LABEL} ({CIFAR10_CLASSES[CLASS_LABEL]})")
print(f"Number of samples tested: {num_samples}")
print(f"Average MSE: {np.mean(all_mses):.6f}")
print(f"Average PSNR: {np.mean(all_psnrs):.2f} dB")
print(f"Best MSE: {np.min(all_mses):.6f} (seed {np.argmin(all_mses)})")
print(f"Best PSNR: {np.max(all_psnrs):.2f} dB (seed {np.argmax(all_psnrs)})")
print()

# Quality assessment
avg_mse = np.mean(all_mses)
if avg_mse < 0.001:
    print("EXCELLENT: Model has perfectly memorized the image!")
elif avg_mse < 0.01:
    print("GOOD: Model has mostly memorized the image.")
elif avg_mse < 0.05:
    print("FAIR: Model is learning but needs more training.")
else:
    print("POOR: Model has not yet memorized the image. Train longer!")

print("="*60)

## Super-Resolution Test (Optional)

Test if the overfitted model can also generate at higher resolution.

In [ ]:
def set_decoder_scale(scale: float):
    """Set NF decoder patch scaling for super-resolution."""
    denoiser.decoder_patch_scaling_h = scale
    denoiser.decoder_patch_scaling_w = scale


@torch.no_grad()
def sample_superres(
    class_label: int,
    height: int = 128,
    width: int = 128,
    seed: int = 42,
    num_steps: int = 50,
):
    """Generate super-resolution sample."""
    torch.manual_seed(seed)
    
    scale = height / 32.0
    set_decoder_scale(scale)
    
    sampler.guidance = 1.0
    sampler.num_steps = num_steps
    
    noise = torch.randn(1, 3, height, width, device=DEVICE)
    
    condition, uncondition = conditioner([class_label])
    condition = condition.to(DEVICE)
    uncondition = uncondition.to(DEVICE)
    
    samples = sampler(
        denoiser,
        noise,
        condition,
        uncondition,
    )
    
    images = vae.decode(samples)
    images = torch.clamp(images, -1.0, 1.0)
    images_uint8 = fp2uint8(images)
    
    # Reset scale
    set_decoder_scale(1.0)
    
    return images_uint8.cpu()


# Generate at different resolutions
print("Generating super-resolution samples...")

best_idx = np.argmin(all_mses)
img_32 = all_samples[best_idx]  # Best 32x32
img_64 = sample_superres(CLASS_LABEL, 64, 64, seed=best_idx, num_steps=100)
img_128 = sample_superres(CLASS_LABEL, 128, 128, seed=best_idx, num_steps=100)

fig, axes = plt.subplots(1, 4, figsize=(16, 4))

axes[0].imshow(target_np)
axes[0].set_title("Target 32x32")
axes[0].axis('off')

axes[1].imshow(img_32.permute(1, 2, 0).numpy())
axes[1].set_title("Generated 32x32")
axes[1].axis('off')

axes[2].imshow(img_64[0].permute(1, 2, 0).numpy())
axes[2].set_title("Super-Res 64x64")
axes[2].axis('off')

axes[3].imshow(img_128[0].permute(1, 2, 0).numpy())
axes[3].set_title("Super-Res 128x128")
axes[3].axis('off')

plt.suptitle("Single Instance at Multiple Resolutions", fontsize=14)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "overfitting_superres.png", dpi=150)
plt.show()

In [ ]:
print("Done!")
print(f"Outputs saved to: {OUTPUT_DIR}")